# Phase 6 — Model Development and Comparison

We train supervised regressors on the offline Kaggle-derived dataset. The target is a transparent proxy constructed from product rating, log review count, and discount percentage. It is a teaching target for regression and model comparison, not a user-labelled measure of personal relevance. Products are grouped by their base title (the text before the first bracket, comma or `|`), so repeated listings and colour/storage variants of the same product cannot cross partitions. We use a development set for model selection and reserve a separate holdout set for Phase 7.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

sys.path.insert(0, "..")
from backend.product_features import FEATURE_COLUMNS, product_feature_frame

DATA_PATH = Path("electronics_feature_engineered.csv")
ARTIFACT_DIR = Path("../backend/model_artifacts")
OUTPUT_DIR = Path("model_outputs")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df), "| Feature count:", len(FEATURE_COLUMNS))

Rows: 5010 | Feature count: 16


## Target definition and split

For each product, rating quality is `rating / 5`; popularity is `log1p(review_count)` divided by the largest log count in the development rows; discount is clipped to 0–100% and scaled to 0–1. The weighted proxy is `0.45 × rating quality + 0.35 × popularity + 0.20 × discount`. Missing ratings would use the development-row median (the source has no missing ratings), missing review counts are treated as zero, and missing discount information receives zero discount contribution. These choices keep all records and are stated as assumptions.

The split is made first, and the two target constants (rating median and maximum log review count) are calculated from development rows only, so the reserved holdout does not influence how the target is built.

**Data caveat:** `review_rating` in this dataset is the star value of one sampled review (always a whole number 1–5), not the product's average rating. The proxy therefore mixes one review's stars with the product's review count and discount.

In [2]:
import re

def product_group(title):
    """Group colour/storage variants of one product together.

    "oneplus nord ce 2 lite 5g (blue tide, 6gb ram)" and
    "oneplus nord ce 2 lite 5g (black dusk, 6gb ram)" both become
    "oneplus nord ce 2 lite 5g", so they always land in the same split.
    """
    text = re.sub(r"\s+", " ", str(title or "").lower()).strip()
    base = re.split(r"[(|,]", text)[0].strip()
    return base or text

df["no_of_ratings"] = pd.to_numeric(df["no_of_ratings"], errors="coerce").fillna(0).clip(lower=0)
X = product_feature_frame(df)
groups = df["name"].map(product_group)
print("Rows:", len(df), "| product groups:", groups.nunique())

# 1) Reserve a final holdout first. It is not used to choose the model.
holdout_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
dev_idx, test_idx = next(holdout_splitter.split(X, groups=groups))

# 2) Build the proxy target. The two constants come from development rows only.
rating = pd.to_numeric(df["review_rating"], errors="coerce")
rating_fill = float(rating.iloc[dev_idx].median())
log_reviews = np.log1p(df["no_of_ratings"])
log_review_max = float(log_reviews.iloc[dev_idx].max()) or 1.0
rating_quality = rating.fillna(rating_fill).clip(0, 5) / 5
popularity = (log_reviews / log_review_max).clip(upper=1)
discount = pd.to_numeric(df["discount_percent"], errors="coerce").clip(0, 100).fillna(0) / 100
y = (0.45 * rating_quality + 0.35 * popularity + 0.20 * discount).astype(float)
print("Target range:", round(float(y.min()), 4), "to", round(float(y.max()), 4))
print("Missing target values:", int(y.isna().sum()))
print("Predictor columns in fixed order:", FEATURE_COLUMNS)

X_dev, X_test = X.iloc[dev_idx], X.iloc[test_idx]
y_dev, y_test = y.iloc[dev_idx], y.iloc[test_idx]
groups_dev = groups.iloc[dev_idx]

# 3) Split the development rows again for model selection.
validation_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=43)
train_rel, val_rel = next(validation_splitter.split(X_dev, groups=groups_dev))
X_train, X_val = X_dev.iloc[train_rel], X_dev.iloc[val_rel]
y_train, y_val = y_dev.iloc[train_rel], y_dev.iloc[val_rel]
print("Model-selection train rows:", len(X_train), "| Validation rows:", len(X_val))
print("Development rows for final fit:", len(X_dev), "| Reserved holdout rows:", len(X_test))
print("Product groups shared by development and holdout:", len(set(groups_dev) & set(groups.iloc[test_idx])))
print("Product groups shared by train and validation:", len(set(groups_dev.iloc[train_rel]) & set(groups_dev.iloc[val_rel])))

Rows: 5010 | product groups: 4155
Target range: 0.1718 to 0.9501
Missing target values: 0
Predictor columns in fixed order: ['price_log1p', 'rating', 'review_count_log1p', 'discount_percent', 'gaming', 'wireless', 'rgb', 'battery_present', 'premium_brand', 'fast_charging', 'ram_gb', 'storage_gb', 'battery_mah', 'screen_inches', 'charging_w', 'refresh_rate_hz']
Model-selection train rows: 3223 | Validation rows: 792
Development rows for final fit: 4015 | Reserved holdout rows: 995
Product groups shared by development and holdout: 0
Product groups shared by train and validation: 0


## Baseline and candidate regressors

The mean baseline predicts the development-training target mean for every validation product. Linear Regression provides an interpretable reference, while a Decision Tree and Random Forest can model nonlinear patterns. Median imputation is fitted only on each training partition inside the pipeline. Candidates are compared on a grouped validation split. The separate holdout remains unused until Phase 7. MAE and RMSE are lower-is-better; R² compares the model with the mean baseline.

In [3]:
models = {
    "Linear Regression": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", LinearRegression())]),
    "Decision Tree": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", DecisionTreeRegressor(max_depth=8, min_samples_leaf=10, random_state=42))]),
    "Random Forest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.8, random_state=42, n_jobs=-1))]),
}

def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred), "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)), "R2": r2_score(y_true, y_pred)}

baseline_pred = np.full(len(y_val), float(y_train.mean()))
results = [{"Model": "Mean Baseline", **metrics(y_val, baseline_pred)}]
for name, model in models.items():
    model.fit(X_train, y_train)
    results.append({"Model": name, **metrics(y_val, model.predict(X_val))})
results_df = pd.DataFrame(results).sort_values(["RMSE", "MAE"]).reset_index(drop=True)
display(results_df.round(4))
results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

candidate = results_df.loc[results_df.Model != "Mean Baseline"].iloc[0]["Model"]
selection_model = models[candidate]
validation_predictions = selection_model.predict(X_val)
print("Selected by lowest validation RMSE:", candidate)
# After selecting hyperparameters/model family, refit on all development rows.
final_model = models[candidate]
final_model.fit(X_dev, y_dev)

,Model,MAE,RMSE,R2
0,Random Forest,0.0066,0.0140,0.9857
1,Decision Tree,0.0159,0.0237,0.9590
2,Linear Regression,0.0120,0.0270,0.9468
3,Mean Baseline,0.0872,0.1171,-0.0002


Selected by lowest validation RMSE: Random Forest


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](16,)","['price_log1p','rating','review_count_log1p',...,'screen_inches', 'charging_w','refresh_rate_hz']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,16
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings 

In [4]:
# Compare model errors and inspect actual versus predicted values for the selected model.
selected_predictions = validation_predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(results_df.Model, results_df.RMSE, color=["#9aa4b2" if name == "Mean Baseline" else "#2f6f8f" for name in results_df.Model])
axes[0].set_ylabel("RMSE")
axes[0].set_title("Validation Error by Model")
axes[0].tick_params(axis="x", rotation=25)
axes[1].scatter(y_val, selected_predictions, alpha=0.35, s=18)
axes[1].plot([0, 1], [0, 1], color="black", linestyle="--")
axes[1].set(xlabel="Actual proxy score", ylabel="Predicted proxy score", title=f"Actual vs predicted: {candidate}")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "phase6_model_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

# Save the exact estimator, input order and target assumptions used for inference.
artifact = {
    "model": final_model,
    "model_name": candidate,
    "feature_columns": FEATURE_COLUMNS,
    "target_definition": "0.45*(rating/5) + 0.35*(log1p(review_count)/training_max_log1p_review_count) + 0.20*(clipped_discount_percent/100)",
    "target_rating_fill": rating_fill,
    "target_log_review_max": log_review_max,
    "random_state": 42,
    "test_size": 0.20,
    "validation_size": 0.20,
    "split_method": "GroupShuffleSplit by base product title (text before the first bracket, comma or |), random states 42 and 43",
    "metrics": results_df.to_dict(orient="records"),
}
joblib.dump(artifact, ARTIFACT_DIR / "product_score.joblib")
print("Saved model artifact:", ARTIFACT_DIR / "product_score.joblib")

Saved model artifact: ../backend/model_artifacts/product_score.joblib


/var/folders/_y/jrtd4g7n1xq8nyb6qbq1yywh0000gn/T/ipykernel_98271/3414218857.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Feature importance describes contribution to the constructed proxy, not causal importance.
estimator = final_model.named_steps["model"]
if hasattr(estimator, "feature_importances_"):
    importance_values = estimator.feature_importances_
elif hasattr(estimator, "coef_"):
    importance_values = np.abs(estimator.coef_)
else:
    importance_values = np.zeros(len(FEATURE_COLUMNS))
importance = pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": importance_values}).sort_values("importance", ascending=False)
display(importance.head(12).round(4))
importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 5))
top = importance.head(12).sort_values("importance")
ax.barh(top.feature, top.importance, color="#2f6f8f")
ax.set(title="Top model feature importances", xlabel="Importance", ylabel="Feature")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "phase6_feature_importance.png", dpi=180, bbox_inches="tight")
plt.show()

,feature,importance
1,rating,0.6082
2,review_count_log1p,0.2438
3,discount_percent,0.1416
0,price_log1p,0.0054
5,wireless,0.0004
8,premium_brand,0.0002
10,ram_gb,0.0001
9,fast_charging,0.0001
13,screen_inches,0.0001
7,battery_present,0.0001


/var/folders/_y/jrtd4g7n1xq8nyb6qbq1yywh0000gn/T/ipykernel_98271/3226759685.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interpretation

The chosen regressor estimates the constructed quality/popularity/discount proxy. Several target components also appear among predictors, so strong validation metrics show that the model reproduces this proxy on unseen product groups. They do not establish personalization, causal effects, or recommendation relevance. Phase 7 checks residuals and train/test differences, then the live ranking layer combines the proxy estimate with explicit query constraints.